# 03 - Pipeline de Streaming con Spark Structured Streaming

Arquitectura Kappa: procesamiento en tiempo real de datos IoT con ventanas, watermarking y escritura en TimescaleDB.

## 1. Configuracion de Spark Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from dotenv import load_dotenv
import os

load_dotenv()

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("IoT-Streaming-Pipeline") \
    .config("spark.sql.streaming.checkpointLocation", "/home/jovyan/checkpoint/iot") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session creada - Version:", spark.version)

Spark Session creada - Version: 3.5.0


## 2. Definicion del Schema del evento

In [3]:
iot_schema = StructType([
    StructField("estacion", StringType(), True),
    StructField("temperatura", DoubleType(), True),
    StructField("humedad", DoubleType(), True),
    StructField("presion", DoubleType(), True),
    StructField("altura", DoubleType(), True),
    StructField("gas", DoubleType(), True),
    StructField("iaq", DoubleType(), True),
    StructField("eco2", DoubleType(), True),
    StructField("VOC", DoubleType(), True),
    StructField("calidad_aire", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("event_timestamp", LongType(), True),
    StructField("delayed", BooleanType(), True),
])

print("Schema definido:")
print(iot_schema.simpleString())

Schema definido:
struct<estacion:string,temperatura:double,humedad:double,presion:double,altura:double,gas:double,iaq:double,eco2:double,VOC:double,calidad_aire:string,created_at:string,event_timestamp:bigint,delayed:boolean>


## 3. Lectura del stream de Kafka

In [4]:
KAFKA_BROKER = "kafka:29092"
TOPIC = "iot.air_quality.streaming"

raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

print("Stream de Kafka configurado correctamente")

Stream de Kafka configurado correctamente


## 4. Parseo y transformacion de datos

In [5]:
parsed_stream = raw_stream.select(
    from_json(col("value").cast("string"), iot_schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select(
    "data.*",
    "kafka_timestamp",
    (col("kafka_timestamp").cast("long") * 1000 - col("event_timestamp")).alias("latencia_ms"),
).withColumn(
    "event_time",
    to_timestamp(col("created_at"))
)

print("Stream parseado y latencia calculada")

Stream parseado y latencia calculada


## 5. Ventanas, Watermarking y Agregaciones

In [6]:
WINDOW_DURATION = "1 minute"
SLIDE_DURATION = "30 seconds"
WATERMARK_DURATION = "30 seconds"

windowed_stream = parsed_stream \
    .withWatermark("event_time", WATERMARK_DURATION) \
    .groupBy(
        window("event_time", WINDOW_DURATION, SLIDE_DURATION),
        "estacion"
    ) \
    .agg(
        round(avg("temperatura"), 2).alias("avg_temperatura"),
        round(avg("humedad"), 2).alias("avg_humedad"),
        round(avg("iaq"), 2).alias("avg_iaq"),
        round(avg("presion"), 2).alias("avg_presion"),
        round(avg("eco2"), 2).alias("avg_eco2"),
        round(avg("VOC"), 3).alias("avg_voc"),
        count("*").alias("evento_count"),
        round(avg("latencia_ms"), 2).alias("latencia_ms")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "estacion",
        "avg_temperatura",
        "avg_humedad",
        "avg_iaq",
        "avg_presion",
        "avg_eco2",
        "avg_voc",
        "evento_count",
        "latencia_ms"
    )

print(f"Ventanas: {WINDOW_DURATION} con slide de {SLIDE_DURATION}")
print(f"Watermark: {WATERMARK_DURATION}")

Ventanas: 1 minute con slide de 30 seconds
Watermark: 30 seconds


## 6. Funcion de escritura en TimescaleDB (foreachBatch)

In [7]:
TSDB_URL = "jdbc:postgresql://timescaledb:5432/iot_metrics"
TSDB_PROPS = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

def write_to_timescaledb(batch_df, batch_id):
    print(f"\n--- Procesando batch {batch_id} con {batch_df.count()} registros ---")
    if batch_df.count() > 0:
        batch_df.write \
            .format("jdbc") \
            .option("url", TSDB_URL) \
            .option("dbtable", "air_quality_metrics") \
            .option("user", TSDB_PROPS["user"]) \
            .option("password", TSDB_PROPS["password"]) \
            .option("driver", TSDB_PROPS["driver"]) \
            .mode("append") \
            .save()
        print(f"Batch {batch_id} escrito en TimescaleDB")
    else:
        print(f"Batch {batch_id} vacio, saltando escritura")

print("Funcion de escritura definida correctamente")

Funcion de escritura definida correctamente


## 7. Inicio del streaming con foreachBatch

In [8]:
query = windowed_stream.writeStream \
    .outputMode("update") \
    .foreachBatch(write_to_timescaledb) \
    .option("checkpointLocation", "/home/jovyan/checkpoint/iot") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Query de streaming iniciada")
print(f"Query ID: {query.id}")
print(f"Estado: {query.isActive}")

Query de streaming iniciada
Query ID: 79da3cea-5079-41e9-8860-2485065f5d3a
Estado: True


## 8. Monitor en consola (debug)

In [9]:
console_query = windowed_stream.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .option("numRows", 5) \
    .trigger(processingTime="10 seconds") \
    .start()

print("Monitor de consola iniciado")

Monitor de consola iniciado


## 9. Verificacion de datos en TimescaleDB

In [12]:
df_tsdb = spark.read \
    .format("jdbc") \
    .option("url", TSDB_URL) \
    .option("dbtable", "air_quality_metrics") \
    .option("user", TSDB_PROPS["user"]) \
    .option("password", TSDB_PROPS["password"]) \
    .option("driver", TSDB_PROPS["driver"]) \
    .load()

print(f"Total registros en TimescaleDB: {df_tsdb.count()}")
df_tsdb.orderBy(col("window_start").desc()).show(10, truncate=False)

Total registros en TimescaleDB: 14
+-------------------+-------------------+--------+---------------+-----------+-------+-----------+--------+-------+------------+-----------+
|window_start       |window_end         |estacion|avg_temperatura|avg_humedad|avg_iaq|avg_presion|avg_eco2|avg_voc|evento_count|latencia_ms|
+-------------------+-------------------+--------+---------------+-----------+-------+-----------+--------+-------+------------+-----------+
|2026-05-24 18:59:00|2026-05-24 19:00:00|ESP32_02|25.0           |59.16      |46.0   |1015.07    |629.0   |0.359  |1           |-919.0     |
|2026-05-24 18:59:00|2026-05-24 19:00:00|ESP32_04|25.67          |58.98      |59.0   |1011.33    |593.0   |0.548  |1           |-919.0     |
|2026-05-24 18:59:00|2026-05-24 19:00:00|ESP32_05|25.51          |57.99      |47.0   |1012.39    |644.0   |0.429  |1           |4081.0     |
|2026-05-24 18:59:00|2026-05-24 19:00:00|ESP32_03|26.03          |55.64      |45.0   |1013.63    |625.0   |0.446  |1   

## 10. Estado del streaming

In [13]:
print("Estado de las queries activas:")
for q in spark.streams.active:
    print(f"  - ID: {q.id} | Activa: {q.isActive} | Nombre: {q.name}")

Estado de las queries activas:
  - ID: 79da3cea-5079-41e9-8860-2485065f5d3a | Activa: True | Nombre: None
  - ID: 0eb56da6-031f-4805-bec1-16afdf2329bb | Activa: True | Nombre: None
Batch 3 escrito en TimescaleDB

--- Procesando batch 4 con 10 registros ---
Batch 4 escrito en TimescaleDB

--- Procesando batch 5 con 10 registros ---
Batch 5 escrito en TimescaleDB

--- Procesando batch 6 con 10 registros ---
Batch 6 escrito en TimescaleDB

--- Procesando batch 7 con 15 registros ---
Batch 7 escrito en TimescaleDB

--- Procesando batch 8 con 15 registros ---
Batch 8 escrito en TimescaleDB

--- Procesando batch 9 con 15 registros ---
Batch 9 escrito en TimescaleDB

--- Procesando batch 10 con 15 registros ---
Batch 10 escrito en TimescaleDB

--- Procesando batch 11 con 15 registros ---
Batch 11 escrito en TimescaleDB

--- Procesando batch 12 con 15 registros ---
Batch 12 escrito en TimescaleDB

--- Procesando batch 13 con 15 registros ---
Batch 13 escrito en TimescaleDB

--- Procesando batc

## 11. Detener queries

In [23]:
for q in spark.streams.active:
    q.stop()
    print(f"Query {q.id} detenida")

spark.stop()

Query 2083b27a-67f9-4988-a54d-f6d7f345f506 detenida
Query a1bd1285-827f-4008-82a7-83f187ccf445 detenida
